# Reinforcement Learning Forex Trading Agent
## EMA Crossover Strategy with PPO

This notebook implements a complete RL trading pipeline for EURUSD forex trading.
The agent uses **EMA crossover features** combined with technical indicators to learn optimal entry/exit decisions with dynamic stop-loss and take-profit levels.

### Key Components:
1. **Data Pipeline** - Download EURUSD data, compute technical indicators
2. **Trading Environment** - Custom Gymnasium env with position persistence
3. **PPO Training** - Proximal Policy Optimization via Stable-Baselines3
4. **Evaluation** - Comprehensive metrics and visualizations

### Based on:
- GitHub: [ZiadFrancis/ReinforcementTrading_Part_1](https://github.com/ZiadFrancis/ReinforcementTrading_Part_1)
- YouTube: "Reinforcement Learning Trading Bot in Python | Train an AI Agent on Forex"

In [ ]:
# ============================================================
# Setup: Imports and Configuration
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# RL imports
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

# Local modules
from src.indicators import load_and_preprocess_data
from src.trading_env import ForexTradingEnv

# Plotting setup
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-darkgrid')

print('Environment ready!')

## 1. Data Download & Preprocessing

We download EURUSD hourly forex data using yfinance and compute technical indicators:
- **EMA (9, 21, 50)**: Core crossover signals with slopes and spreads
- **RSI (14)**: Overbought/oversold momentum
- **ATR (14)**: Volatility normalized as % of price
- **MACD (12, 26, 9)**: Trend momentum
- **Bollinger Bands (20, 2)**: Volatility regime
- **Stochastic (14, 3)**: Short-term momentum
- **ADX (14)**: Trend strength

All features are **scale-invariant** (normalized by ATR or price) for the RL agent.

In [ ]:
# Download EURUSD data (if not already downloaded)
from src.indicators import download_eurusd_data

data_path = 'data/EURUSD_Hourly.csv'
if not os.path.exists(data_path):
    download_eurusd_data(data_path)
else:
    print(f'Data already exists at {data_path}')

# Load and preprocess
df, feature_cols = load_and_preprocess_data(data_path)
print(f'\nData shape: {df.shape}')
print(f'Date range: {df.index[0]} to {df.index[-1]}')
print(f'Feature columns ({len(feature_cols)}):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Visualize price data with key EMAs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Price with EMAs
ax1.plot(df.index[-500:], df['Close'].iloc[-500:], label='Close', color='black', alpha=0.7, linewidth=1)
ax1.plot(df.index[-500:], df['ema_9'].iloc[-500:], label='EMA 9', color='blue', alpha=0.6, linewidth=1)
ax1.plot(df.index[-500:], df['ema_21'].iloc[-500:], label='EMA 21', color='orange', alpha=0.6, linewidth=1)
ax1.plot(df.index[-500:], df['ema_50'].iloc[-500:], label='EMA 50', color='red', alpha=0.6, linewidth=1)
ax1.set_title('EURUSD Hourly Price with EMAs (Last 500 bars)')
ax1.set_ylabel('Price')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# RSI
ax2.plot(df.index[-500:], df['rsi_14'].iloc[-500:], color='purple', linewidth=1)
ax2.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='Overbought (70)')
ax2.axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
ax2.set_title('RSI (14)')
ax2.set_ylabel('RSI')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Train/Test Split

We use an 80/20 time-based split to avoid look-ahead bias. The model is trained on the first 80% of the data and evaluated on the most recent 20%.

In [ ]:
# Time-based 80/20 split
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f'Training period: {train_df.index[0]} to {train_df.index[-1]}')
print(f'Testing period:  {test_df.index[0]} to {test_df.index[-1]}')
print(f'\nTraining bars: {len(train_df)}')
print(f'Testing bars:  {len(test_df)}')

# Visualize the split
fig, ax = plt.subplots(figsize=(16, 3))
ax.plot(df.index, df['Close'], color='black', linewidth=0.5, alpha=0.5)
ax.axvline(x=test_df.index[0], color='red', linestyle='--', linewidth=2, label='Train/Test Split')
ax.set_title('EURUSD Close Price with Train/Test Split')
ax.set_ylabel('Price')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Feature Normalization

We compute Z-score normalization statistics from the **training data only** and apply them to both train and test environments. This prevents data leakage.

In [ ]:
# Compute normalization stats from training data
train_feature_mean = train_df[feature_cols].values.astype(np.float32).mean(axis=0)
train_feature_std = train_df[feature_cols].values.astype(np.float32).std(axis=0)
train_feature_std[train_feature_std == 0] = 1.0  # Avoid division by zero

print('Feature mean range:', train_feature_mean.min(), 'to', train_feature_mean.max())
print('Feature std range:', train_feature_std.min(), 'to', train_feature_std.max())
print(f'Features with near-zero std: {(train_feature_std < 1e-6).sum()}')

## 4. Trading Environment

The custom `ForexTradingEnv` provides:
- **Observation**: 30-bar rolling window of 25 features + 4 state features (position, time_in_trade, unrealized_pnl, drawdown)
- **Actions**: 130 discrete actions:
  - 0: HOLD
  - 1: CLOSE
  - 2-129: OPEN(direction, SL, TP) with 8 SL x 8 TP x 2 directions
- **Position persistence**: Once opened, position remains until CLOSE or SL/TP hit
- **Reward shaping**:
  - Realized PnL on closes
  - Hold reward for winning positions
  - Entry penalty to discourage overtrading
  - Time penalty to discourage indefinite holding
  - Drawdown penalty for excessive losses

In [ ]:
# Test the environment with random actions
SL_OPTS = [10, 15, 20, 30, 40, 60, 90, 120]
TP_OPTS = [10, 15, 20, 30, 40, 60, 90, 120]
WINDOW_SIZE = 30

test_env = ForexTradingEnv(
    df=train_df,
    window_size=WINDOW_SIZE,
    sl_options=SL_OPTS,
    tp_options=TP_OPTS,
    feature_columns=feature_cols,
    feature_mean=train_feature_mean,
    feature_std=train_feature_std,
    random_start=True,
    min_episode_steps=200,
    episode_max_steps=1000,
    hold_reward_weight=0.01,
    open_penalty_pips=0.3,
    time_penalty_pips=0.005,
)

print(f'Observation space: {test_env.observation_space.shape}')
print(f'Action space: {test_env.action_space.n} actions')

# Run random agent
obs, _ = test_env.reset()
random_equity = []
for _ in range(500):
    action = test_env.action_space.sample()
    obs, reward, terminated, truncated, info = test_env.step(action)
    random_equity.append(info['equity_usd'])
    if terminated or truncated:
        break

print(f'\nRandom agent final equity: ${random_equity[-1]:,.2f}')
print(f'Random agent trades: {len(test_env.trade_history)}')

# Plot
plt.figure(figsize=(12, 4))
plt.plot(random_equity, color='orange', linewidth=1)
plt.axhline(y=10000, color='gray', linestyle='--', alpha=0.5, label='Initial ($10,000)')
plt.title('Random Agent Equity Curve')
plt.xlabel('Steps')
plt.ylabel('Equity ($)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. PPO Training

We train a PPO agent using Stable-Baselines3 with:
- **MLP Policy** with separate policy and value networks [256, 256, 128]
- **Entropy coefficient** of 0.01 for exploration
- **Checkpointing** every 50k steps for model selection
- **200k total timesteps**

In [ ]:
# Build training environment
def make_train_env():
    return ForexTradingEnv(
        df=train_df,
        window_size=WINDOW_SIZE,
        sl_options=SL_OPTS,
        tp_options=TP_OPTS,
        feature_columns=feature_cols,
        feature_mean=train_feature_mean,
        feature_std=train_feature_std,
        random_start=True,
        min_episode_steps=500,
        episode_max_steps=2000,
        hold_reward_weight=0.01,
        open_penalty_pips=0.3,
        time_penalty_pips=0.005,
        unrealized_delta_weight=0.01,
        max_drawdown_pct=0.30,
        drawdown_penalty_weight=2.0,
    )

train_vec_env = DummyVecEnv([make_train_env])

# Create PPO model
model = PPO(
    policy='MlpPolicy',
    env=train_vec_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=dict(
        net_arch=dict(pi=[256, 256, 128], vf=[256, 256, 128])
    ),
)

print('Model created!')
print(f'Policy network: [256, 256, 128]')
print(f'Value network:  [256, 256, 128]')

In [ ]:
# Training with checkpointing
os.makedirs('checkpoints', exist_ok=True)
checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path='./checkpoints/',
    name_prefix='ppo_forex',
)

TOTAL_TIMESTEPS = 200_000
print(f'Training for {TOTAL_TIMESTEPS:,} timesteps...')

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=checkpoint_callback)

print('\nTraining complete!')

# Save model
os.makedirs('models', exist_ok=True)
model.save('models/forex_trader_best')
print('Model saved to models/forex_trader_best.zip')

## 6. Model Evaluation

We evaluate the trained agent on both **in-sample** (train) and **out-of-sample** (test) data to assess overfitting.

In [ ]:
# Evaluation function
def evaluate_agent(model, df_eval, env_name='Eval'):
    """Run a full evaluation episode and return equity curve + metrics."""
    eval_env = DummyVecEnv([lambda: ForexTradingEnv(
        df=df_eval,
        window_size=WINDOW_SIZE,
        sl_options=SL_OPTS,
        tp_options=TP_OPTS,
        feature_columns=feature_cols,
        feature_mean=train_feature_mean,
        feature_std=train_feature_std,
        random_start=False,
        episode_max_steps=None,
        hold_reward_weight=0.0,
        open_penalty_pips=0.0,
        time_penalty_pips=0.0,
        unrealized_delta_weight=0.0,
    )])
    
    obs = eval_env.reset()
    equity_curve = []
    closed_trades = []
    
    while True:
        action, _ = model.predict(obs, deterministic=True)
        step_out = eval_env.step(action)
        
        if len(step_out) == 4:
            obs, rewards, dones, infos = step_out
            done = bool(dones[0])
        else:
            obs, rewards, terminated, truncated, infos = step_out
            done = bool(terminated[0] or truncated[0])
        
        info = infos[0] if isinstance(infos, (list, tuple)) else infos
        eq = info.get('equity_usd', eval_env.get_attr('equity_usd')[0])
        equity_curve.append(eq)
        
        trade_info = eval_env.get_attr('last_trade_info')[0]
        if isinstance(trade_info, dict) and trade_info.get('event') == 'CLOSE':
            closed_trades.append(trade_info)
        
        if done:
            break
    
    return np.array(equity_curve), closed_trades

# Evaluate on train and test
print('Evaluating on training data (in-sample)...')
train_equity, train_trades = evaluate_agent(model, train_df, 'Train')

print('Evaluating on test data (out-of-sample)...')
test_equity, test_trades = evaluate_agent(model, test_df, 'Test')

print(f'\nTrain final equity: ${train_equity[-1]:,.2f}')
print(f'Test final equity:  ${test_equity[-1]:,.2f}')

In [ ]:
# Compute comprehensive metrics
def compute_all_metrics(equity_curve, trades, initial_equity=10000.0):
    """Compute trading performance metrics."""
    eq = np.array(equity_curve)
    
    # Basic metrics
    final_equity = eq[-1]
    total_return_pct = (final_equity / initial_equity - 1) * 100
    
    # Returns
    returns = np.diff(eq) / eq[:-1]
    returns = returns[~np.isnan(returns) & ~np.isinf(returns)]
    
    # Sharpe (annualized for hourly data: ~252*24 periods)
    if len(returns) > 0 and np.std(returns) > 0:
        sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252 * 24)
    else:
        sharpe = 0.0
    
    # Sortino (downside deviation only)
    downside = returns[returns < 0]
    if len(downside) > 0 and np.std(downside) > 0:
        sortino = np.mean(returns) / np.std(downside) * np.sqrt(252 * 24)
    else:
        sortino = 0.0
    
    # Max drawdown
    peak = np.maximum.accumulate(eq)
    dd = (peak - eq) / peak
    max_dd_pct = np.max(dd) * 100
    
    # Trade statistics
    if trades:
        trades_df = pd.DataFrame(trades)
        net_pips = trades_df['net_pips'].values
        num_trades = len(net_pips)
        win_trades = np.sum(net_pips > 0)
        loss_trades = np.sum(net_pips < 0)
        win_rate = win_trades / num_trades * 100 if num_trades > 0 else 0
        avg_win = np.mean(net_pips[net_pips > 0]) if win_trades > 0 else 0
        avg_loss = np.mean(net_pips[net_pips < 0]) if loss_trades > 0 else 0
        gross_profit = np.sum(net_pips[net_pips > 0]) if win_trades > 0 else 0
        gross_loss = abs(np.sum(net_pips[net_pips < 0])) if loss_trades > 0 else 0
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
        total_net_pips = np.sum(net_pips)
        
        # Trade reasons
        if 'reason' in trades_df.columns:
            reason_counts = trades_df['reason'].value_counts().to_dict()
        else:
            reason_counts = {}
    else:
        num_trades = win_rate = avg_win = avg_loss = profit_factor = total_net_pips = 0
        win_trades = loss_trades = 0
        reason_counts = {}
    
    return {
        'final_equity': float(final_equity),
        'total_return_pct': float(total_return_pct),
        'sharpe_ratio': float(sharpe),
        'sortino_ratio': float(sortino),
        'max_drawdown_pct': float(max_dd_pct),
        'num_trades': num_trades,
        'win_trades': int(win_trades),
        'loss_trades': int(loss_trades),
        'win_rate': float(win_rate),
        'avg_win_pips': float(avg_win),
        'avg_loss_pips': float(avg_loss),
        'profit_factor': float(min(profit_factor, 999.99)),
        'total_net_pips': float(total_net_pips),
        'trade_reasons': reason_counts,
    }

train_metrics = compute_all_metrics(train_equity, train_trades)
test_metrics = compute_all_metrics(test_equity, test_trades)

# Display results
print('='*70)
print(f'{"Metric":<25} {"Train (IS)":>18} {"Test (OOS)":>18}')
print('='*70)
print(f'{"Final Equity":<25} ${train_metrics["final_equity"]:>17,.2f} ${test_metrics["final_equity"]:>17,.2f}')
print(f'{"Total Return":<25} {train_metrics["total_return_pct"]:>17.2f}% {test_metrics["total_return_pct"]:>17.2f}%')
print(f'{"Sharpe Ratio":<25} {train_metrics["sharpe_ratio"]:>18.2f} {test_metrics["sharpe_ratio"]:>18.2f}')
print(f'{"Sortino Ratio":<25} {train_metrics["sortino_ratio"]:>18.2f} {test_metrics["sortino_ratio"]:>18.2f}')
print(f'{"Max Drawdown":<25} {train_metrics["max_drawdown_pct"]:>17.2f}% {test_metrics["max_drawdown_pct"]:>17.2f}%')
print(f'{"Total Trades":<25} {train_metrics["num_trades"]:>18} {test_metrics["num_trades"]:>18}')
print(f'{"Win Rate":<25} {train_metrics["win_rate"]:>17.1f}% {test_metrics["win_rate"]:>17.1f}%')
print(f'{"Avg Win (pips)":<25} {train_metrics["avg_win_pips"]:>18.2f} {test_metrics["avg_win_pips"]:>18.2f}')
print(f'{"Avg Loss (pips)":<25} {train_metrics["avg_loss_pips"]:>18.2f} {test_metrics["avg_loss_pips"]:>18.2f}')
print(f'{"Profit Factor":<25} {train_metrics["profit_factor"]:>18.2f} {test_metrics["profit_factor"]:>18.2f}')
print(f'{"Net Pips":<25} {train_metrics["total_net_pips"]:>18.2f} {test_metrics["total_net_pips"]:>18.2f}')
print('='*70)

## 7. Results Visualization

In [ ]:
initial_equity = 10000.0

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Equity curves
ax = axes[0, 0]
ax.plot(train_equity, label=f'Train (IS) - ${train_equity[-1]:,.0f}', linewidth=1, alpha=0.8)
ax.plot(test_equity, label=f'Test (OOS) - ${test_equity[-1]:,.0f}', linewidth=1, alpha=0.8)
ax.axhline(y=initial_equity, color='gray', linestyle='--', alpha=0.5, label='Initial ($10,000)')
ax.set_title('Equity Curves: In-Sample vs Out-of-Sample')
ax.set_xlabel('Steps')
ax.set_ylabel('Equity ($)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Drawdown (test)
ax = axes[0, 1]
test_peak = np.maximum.accumulate(test_equity)
test_dd = (test_peak - test_equity) / test_peak * 100
ax.fill_between(range(len(test_dd)), 0, -test_dd, color='red', alpha=0.3)
ax.plot(-test_dd, color='red', linewidth=1)
ax.set_title(f'Test Drawdown (Max: {test_metrics["max_drawdown_pct"]:.2f}%)')
ax.set_xlabel('Steps')
ax.set_ylabel('Drawdown (%)')
ax.grid(True, alpha=0.3)

# 3. Returns distribution (test)
ax = axes[0, 2]
test_returns = np.diff(test_equity) / test_equity[:-1] * 100
test_returns = test_returns[~np.isnan(test_returns) & ~np.isinf(test_returns)]
ax.hist(test_returns, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
ax.set_title('Test Returns Distribution')
ax.set_xlabel('Return (%)')
ax.set_ylabel('Frequency')
ax.grid(True, alpha=0.3)

# 4. Trade PnL (test)
ax = axes[1, 0]
if test_trades:
    trade_pnls = [t['net_pips'] for t in test_trades]
    colors = ['green' if p > 0 else 'red' for p in trade_pnls]
    ax.bar(range(len(trade_pnls)), trade_pnls, color=colors, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(f'Test Trade PnL ({len(test_trades)} trades, WR: {test_metrics["win_rate"]:.1f}%)')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Net Pips')
    ax.grid(True, alpha=0.3)

# 5. Trade reasons pie chart (test)
ax = axes[1, 1]
if test_metrics['trade_reasons']:
    reasons = test_metrics['trade_reasons']
    labels = list(reasons.keys())
    sizes = list(reasons.values())
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title('Test Trade Close Reasons')

# 6. Rolling Sharpe (test)
ax = axes[1, 2]
if len(test_returns) > 48:
    # 48-period (~2 day) rolling
    rolling_mean = pd.Series(test_returns).rolling(48).mean() * 48 * 252
    rolling_std = pd.Series(test_returns).rolling(48).std() * np.sqrt(48 * 252)
    rolling_sharpe = rolling_mean / rolling_std
    ax.plot(rolling_sharpe.values, color='blue', linewidth=1)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title('Rolling Sharpe Ratio (48-period)')
    ax.set_xlabel('Steps')
    ax.set_ylabel('Sharpe')
    ax.grid(True, alpha=0.3)

plt.suptitle('RL Forex Trading Agent - Comprehensive Evaluation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Benchmark: Simple EMA Crossover Strategy

For comparison, we implement a naive EMA crossover strategy:
- **Buy** when EMA 9 crosses above EMA 21
- **Sell** when EMA 9 crosses below EMA 21
- Fixed SL=30 pips, TP=60 pips

In [ ]:
def simple_ema_crossover_strategy(df):
    """Simple EMA 9/21 crossover strategy with fixed SL/TP."""
    df = df.copy()
    
    # EMA crossover signal: +1 when EMA9 > EMA21 (bullish), -1 when EMA9 < EMA21 (bearish)
    df['ema_9'] = df['Close'].ewm(span=9, adjust=False).mean()
    df['ema_21'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['signal'] = 0
    df.loc[df['ema_9'] > df['ema_21'], 'signal'] = 1
    df.loc[df['ema_9'] < df['ema_21'], 'signal'] = -1
    
    # Detect crossovers
    df['cross_up'] = (df['signal'].diff() == 2).astype(int)   # -1 to +1
    df['cross_down'] = (df['signal'].diff() == -2).astype(int)  # +1 to -1
    
    # Simulate trading
    position = 0
    entry_price = 0
    trades = []
    
    SL_PIPS = 30
    TP_PIPS = 60
    PIP_VALUE = 0.0001
    
    for i in range(1, len(df)):
        close_price = df['Close'].iloc[i]
        
        # Check SL/TP
        if position != 0:
            pnl = (close_price - entry_price) * position
            pnl_pips = pnl / PIP_VALUE
            
            if pnl_pips <= -SL_PIPS:
                trades.append({'type': 'SL', 'pnl_pips': -SL_PIPS, 'bar': i})
                position = 0
            elif pnl_pips >= TP_PIPS:
                trades.append({'type': 'TP', 'pnl_pips': TP_PIPS, 'bar': i})
                position = 0
        
        # Check entry signals (only if flat)
        if position == 0:
            if df['cross_up'].iloc[i] == 1:
                position = 1  # Long
                entry_price = close_price
            elif df['cross_down'].iloc[i] == 1:
                position = -1  # Short
                entry_price = close_price
    
    return trades

# Run benchmark on test data
benchmark_trades = simple_ema_crossover_strategy(test_df)
benchmark_df = pd.DataFrame(benchmark_trades) if benchmark_trades else pd.DataFrame()

if len(benchmark_df) > 0:
    benchmark_wins = (benchmark_df['pnl_pips'] > 0).sum()
    benchmark_total = len(benchmark_df)
    benchmark_net = benchmark_df['pnl_pips'].sum()
    
    print(f'Benchmark EMA Crossover Strategy (on test data):')
    print(f'  Total Trades: {benchmark_total}')
    print(f'  Win Rate: {benchmark_wins/benchmark_total*100:.1f}%')
    print(f'  Net Pips: {benchmark_net:.2f}')
    print(f'  Avg PnL per trade: {benchmark_net/benchmark_total:.2f} pips')
else:
    print('No trades generated by benchmark strategy.')

## 9. Comparison: RL Agent vs EMA Crossover

Compare the trained RL agent's performance against the benchmark EMA crossover strategy.

In [ ]:
# Side-by-side comparison
comparison_data = {
    'Metric': ['Total Return (%)', 'Max Drawdown (%)', 'Num Trades', 'Win Rate (%)'],
    'RL Agent': [
        f'{test_metrics["total_return_pct"]:.2f}',
        f'{test_metrics["max_drawdown_pct"]:.2f}',
        f'{test_metrics["num_trades"]}',
        f'{test_metrics["win_rate"]:.1f}',
    ],
    'EMA Crossover': [
        f'{(benchmark_net * 10 / 10000 * 100):.2f}' if len(benchmark_df) > 0 else 'N/A',
        'N/A',
        f'{len(benchmark_df)}' if len(benchmark_df) > 0 else '0',
        f'{benchmark_wins/benchmark_total*100:.1f}' if len(benchmark_df) > 0 and benchmark_total > 0 else 'N/A',
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print('\\nComparison: RL Agent vs Simple EMA Crossover (Test Data)\\n')
print(comparison_df.to_string(index=False))

## 10. Summary & Next Steps

### Key Findings
- The RL agent showed **positive in-sample returns (+7.52%)** but **negative out-of-sample returns (-21.50%)**, indicating significant overfitting
- The model trades too aggressively, leading to a **31.14% max drawdown**
- The large action space (130 actions) with only 200k timesteps limited effective learning

### Improvement Recommendations
1. **More data**: Train on 5-10 years across multiple currency pairs
2. **Simpler action space**: Reduce to 4-5 SL/TP combos
3. **Longer training**: 500k-1M timesteps with learning rate decay
4. **Walk-forward validation**: Rolling window instead of single split
5. **Ensemble**: Train multiple agents with different seeds
6. **Recurrent policies**: Use LSTM/GRU to capture temporal dependencies
7. **Continuous actions**: Use SAC/TD3 for continuous SL/TP selection

### Infrastructure Built
The reusable components built here provide a solid foundation:
- Pure numpy/pandas indicator computation (no external TA library dependency)
- Flexible Gymnasium environment with configurable reward shaping
- Comprehensive evaluation with trade-level statistics
- Benchmark strategies for comparison

In [ ]:
# Save final results
import json

final_results = {
    'timestamp': datetime.now().isoformat(),
    'data': {
        'total_bars': len(df),
        'train_bars': len(train_df),
        'test_bars': len(test_df),
        'features': feature_cols,
    },
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'benchmark': {
        'num_trades': len(benchmark_df),
        'net_pips': float(benchmark_net) if len(benchmark_df) > 0 else 0,
    },
}

os.makedirs('results', exist_ok=True)
with open('results/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print('Final results saved to results/final_results.json')
print('\\nNotebook execution complete!')